In [8]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score


df = pd.read_csv("../../data/Metadata_features/metadata_features.csv")

X = df[['lesion_green_share', 'saturation_variance', 'melanoma_color_count',
       'mabrouk_asymmetry_score', 'avg_asymmetry_score', 'worst_score',
       'convexity_score', 'lesion_red_share', 'lesion_skin_green_diff',
       'Polsby-Popper']]
y = df["skin_cancer_diagnosis"]


patient_labels = df.groupby("patient_id")["skin_cancer_diagnosis"].max()

patients = patient_labels.index
labels = patient_labels.values

train_patients, test_patients = train_test_split(
    patients,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

train_mask = df["patient_id"].isin(train_patients)

X_train = X[train_mask]
y_train = y[train_mask]

groups_train = df.loc[X_train.index, "patient_id"]

# Parameter sweep
weights = np.arange(1, 10, 0.5)

train_accs = []
val_accs = []
train_aucs = []
val_aucs = []

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for w in weights:

    train_acc_scores = []
    val_acc_scores = []
    train_auc_scores = []
    val_auc_scores = []

    for train_idx, val_idx in cv.split(X_train, y_train, groups_train):

        X_tr = X_train.iloc[train_idx]
        X_val_fold = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val_fold = y_train.iloc[val_idx]

        # Scaling
        scaler = StandardScaler()
        X_tr_scaled = scaler.fit_transform(X_tr)
        X_val_scaled = scaler.transform(X_val_fold)

        model = LogisticRegression(
            random_state=42,
            class_weight={0: 1, 1: w},
            max_iter=1000
        )

        model.fit(X_tr_scaled, y_tr)

        # Accuracy
        y_tr_pred = model.predict(X_tr_scaled)
        y_val_pred = model.predict(X_val_scaled)

        train_acc_scores.append(accuracy_score(y_tr, y_tr_pred))
        val_acc_scores.append(accuracy_score(y_val_fold, y_val_pred))

        # AUC
        y_tr_proba = model.predict_proba(X_tr_scaled)[:, 1]
        y_val_proba = model.predict_proba(X_val_scaled)[:, 1]

        train_auc_scores.append(roc_auc_score(y_tr, y_tr_proba))
        val_auc_scores.append(roc_auc_score(y_val_fold, y_val_proba))

    # Store means
    train_accs.append(np.mean(train_acc_scores))
    val_accs.append(np.mean(val_acc_scores))
    train_aucs.append(np.mean(train_auc_scores))
    val_aucs.append(np.mean(val_auc_scores))

    print(
        f"class_weight_1 = {w:.1f} "
        f"- Train Acc: {train_accs[-1]:.4f} - Val Acc: {val_accs[-1]:.4f} "
        f"- Train AUC: {train_aucs[-1]:.4f} - Val AUC: {val_aucs[-1]:.4f}"
    )

class_weight_1 = 1.0 - Train Acc: 0.7100 - Val Acc: 0.7084 - Train AUC: 0.7856 - Val AUC: 0.7789
class_weight_1 = 1.5 - Train Acc: 0.6971 - Val Acc: 0.6922 - Train AUC: 0.7856 - Val AUC: 0.7790
class_weight_1 = 2.0 - Train Acc: 0.6872 - Val Acc: 0.6804 - Train AUC: 0.7855 - Val AUC: 0.7792
class_weight_1 = 2.5 - Train Acc: 0.6634 - Val Acc: 0.6581 - Train AUC: 0.7854 - Val AUC: 0.7791
class_weight_1 = 3.0 - Train Acc: 0.6416 - Val Acc: 0.6379 - Train AUC: 0.7854 - Val AUC: 0.7790
class_weight_1 = 3.5 - Train Acc: 0.6247 - Val Acc: 0.6191 - Train AUC: 0.7853 - Val AUC: 0.7790
class_weight_1 = 4.0 - Train Acc: 0.6064 - Val Acc: 0.6005 - Train AUC: 0.7852 - Val AUC: 0.7791
class_weight_1 = 4.5 - Train Acc: 0.5891 - Val Acc: 0.5881 - Train AUC: 0.7852 - Val AUC: 0.7790
class_weight_1 = 5.0 - Train Acc: 0.5774 - Val Acc: 0.5773 - Train AUC: 0.7851 - Val AUC: 0.7789
class_weight_1 = 5.5 - Train Acc: 0.5701 - Val Acc: 0.5677 - Train AUC: 0.7851 - Val AUC: 0.7789
class_weight_1 = 6.0 - Train A

In [ ]:
import plotly.graph_objects as go


df = pd.read_csv("../../data/Metadata_features/metadata_features.csv")

df = df.drop(["img_id", "gender", "diagnostic", "biopsed", "melanoma_colors"], axis = 1)
df["prediction"] = False

x = df.iloc[:, 3:]
y = df['skin_cancer_diagnosis']

x_train, x_valtest, y_train, y_valtest = train_test_split(
x, y, test_size = 0.4, random_state = 8008135, stratify = y)

x_val, x_test, y_val, y_test = train_test_split(
x_valtest, y_valtest, test_size = 0.5, random_state = 8008135, stratify = y_valtest)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_val_scaled = scaler.transform(x_val)
x_test_scaled = scaler.transform(x_test)

x = np.arange(0, 1, 0.01)
y = np.arange(0, 10, 0.1)
models = [LogisticRegression(random_state = 8008135, class_weight = {1: i, 0: 1}) for i in y]
for model in models:
    model.fit(x_train_scaled, y_train) 
preds = [model.predict_proba(x_val_scaled)[:, 1] for model in models]
z = np.array([[accuracy_score(y_val, (j >= i).astype(int)) for i in x] for j in preds])

fig = go.Figure()
fig.add_trace(go.Surface(x = x, y = y, z = z))

In [41]:
max_auc_pos = np.unravel_index(np.argmax(z), z.shape)
print(f"Best AUC is {z.max()} and has weight {max_auc_pos[0]} and threshold {max_auc_pos[1]/100}")

Best AUC is 0.7374701670644391 and has weight 1 and threshold 0.09
